## 1. Import libraries and configure the inference environment

In [1]:
import sys
from pathlib import Path
_ROOT = Path.cwd().parent.resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from    typing      import  Literal, Sequence
from    itertools   import  product

from    tqdm.notebook   import  tqdm
from    time            import  perf_counter

import  torch
from    torch.utils.data                import  DataLoader

from    core.utils            import  GaussianNormalizer
from    core.neural.operator  import  PFNO

from    loader      import  NormKeys, HistKeys
from    loader      import  path_base
from    loader      import  load_model_args, load_model_data
from    dataset     import  nonparametric_dataset, parametric_dataset
from    plot        import  *


DEVICE: torch.device = torch.device("cuda:1")

list_max_freq   = [6, 8, 10]
list_seed       = [0, 1, 2]
max_freq        = list_max_freq[1]
seed            = list_seed[0]

res_v = 2**7

## 2. Instantiate the trained operators and normalizers

### 2-1. Load data (trained parameters, normalizers, and training histories)

In [2]:
model_data_pfno     = load_model_data("PFNO2D-param", max_freq, seed)

### 2-2. Load trained operators

In [3]:
model_pfno  = PFNO(**load_model_args("PFNO2D-param", max_freq))

model_pfno.load_state_dict( model_data_pfno["state_dict"])
model_pfno.to(DEVICE)
model_pfno.eval()
print(f"Successfully loaded models.")

Successfully loaded models.


### 2-4. Load normalizers

In [4]:
norm_pfno:  dict[NormKeys, GaussianNormalizer] = model_data_pfno["normalizer"]
for v in norm_pfno.values():  v.to(DEVICE)
print(f"Successfully loaded normalizers.")

Successfully loaded normalizers.


## 3. Conduct inference

### 3-1. Construct dataloader

In [5]:
path_data = path_base / "datasets" / "test" / f"Boltzmann_2D__coeff1.59e-01__res_t051_v{res_v:03d}"
assert path_data.exists(), f"The data path {path_data} does not exist."
list_alpha = [0.50]
part_start, part_end = 1, 1

data, params = [], []
generation_time = 0.0
files = sorted(path_data.glob("alpha*.pth"))
alpha_and_part = list(product(list_alpha, range(part_start, part_end+1)))
for alpha, part in tqdm(alpha_and_part):
    if res_v>2**6:
        file = path_data / f"alpha{alpha:.2f}__part{part:03d}.pth"
    else:
        file = path_data / f"alpha{alpha:.2f}__part{part:02d}.pth"
    f = torch.load(file, weights_only=False)
    data.append(f['data'][[0]])         # (num_instances, res_t, 1, 1, res, res, 1)
    params.append(f['vhs_alpha']*torch.ones(1, 1))     # float
    generation_time += f['elapsed_time']
data    = torch.cat(data, dim=0).to(DEVICE)               # (num_total_instances, res_t, 1, 1, res, res, 1)
params  = torch.cat(params, dim=0).to(DEVICE)           # (num_total_instances, 1)
num_inst, res_t, res_v = data.size(0), data.size(1), data.size(-2)
data    = data.view(num_inst, res_t, res_v, res_v, 1)
data    = data.type(torch.float)
params  = params.type(torch.float)

print(f"Successfully loaded dataset with {num_inst} instances.")

  0%|          | 0/1 [00:00<?, ?it/s]

Successfully loaded dataset with 1 instances.


In [6]:
BATCH_SIZE = 2**6

dataloader_pfno = DataLoader(parametric_dataset(data, params, norm_pfno), batch_size=BATCH_SIZE, shuffle=False)
print(f"Successfully prepared dataloaders.")

Successfully prepared dataloaders.


### 3-2. Compute the trajectories on the test set

In [7]:
torch.cuda.reset_peak_memory_stats(DEVICE)
torch.cuda.synchronize(DEVICE)
pred_pfno = []

with torch.inference_mode():
    model_preds = []
    traj:   torch.Tensor
    alphas: torch.Tensor
    for traj, alphas in tqdm(dataloader_pfno, desc="PFNO Inference"):
        traj    = traj.to(DEVICE)
        alphas  = alphas.to(DEVICE)
        pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
        # Save the initial condition
        batch_pred = [norm_pfno['data'].decode(pred).cpu()]
        # Compute the trajectory
        for t in range(1, res_t):
            pred = model_pfno.forward(pred, alphas)
            batch_pred.append(norm_pfno['data'].decode(pred).cpu())
        batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
        model_preds.append(batch_pred)
    pred_pfno = torch.cat(model_preds, dim=0)   # (num_inst, res_t, res, res, 1)

torch.cuda.synchronize(DEVICE)
peak_memory_pfno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
print(f"PFNO Peak memory usage: {peak_memory_pfno:.2f}MB.")

PFNO Inference:   0%|          | 0/1 [00:00<?, ?it/s]

PFNO Peak memory usage: 49.83MB.


End of file